# Pixels to Predictions: DL Vision Challenge

**Model:** `HuggingFaceTB/SmolVLM-500M-Instruct`  
**Fine-tuning:** QLoRA (4-bit NF4, ≤5M trainable params)  
**Scoring:** Multiple-choice log-likelihood

---

## 📓 Experiment Diary — Run Start

> **Pre-run notes:**

In [ ]:
RUN_ID = "run_1"

In [ ]:
import datetime
import os, json
stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
os.makedirs("diary", exist_ok=True)
with open("diary/compute_diary.txt", "a") as f:
    f.write(json.dumps({"event": "timestamp", "when": stamp}) + "\n")
print("⏱️ Compute diary timestamp:", stamp)

⏱️ Compute diary timestamp: 2026-04-30 21:38:52


## 0. Imports & Environment

In [ ]:
# ── 0a. Install dependencies (uncomment once per environment) ─────────────────
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.3 MB/s eta 0:00:00


In [ ]:
import os
import json
import math
import time
import random
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType

print("All imports OK.")

All imports OK.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR_PATH = '/content/drive/MyDrive/Colab Notebooks/' + RUN_ID + '/'
DRIVE_DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/pixels-to-predictions.zip'

# Paths — adjust to Colab layout

# Local path for faster access
LOCAL_DATA_DIR = Path("/content/data")

# Do not recopy if data already exists
if not os.path.exists(LOCAL_DATA_DIR):
    # Ensure local data directory exists
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    # Copy data from Drive to local disk
    print(f"Copying data from {DRIVE_DATA_PATH} to disk...")
    !cp -r "{DRIVE_DATA_PATH}" "."
    !unzip -q "pixels-to-predictions.zip" -d "{LOCAL_DATA_DIR}"
    print("Data copy complete.")

# Use the local data directory for the rest of the notebook
DATA_DIR = LOCAL_DATA_DIR / Path("pixels-to-predictions/")
print(f'All data present at: {DATA_DIR}')

Mounted at /content/drive
Copying data from /content/drive/MyDrive/Colab Notebooks/pixels-to-predictions.zip to disk...
Data copy complete.
All data present at: /content/data/pixels-to-predictions


In [ ]:
# ── 0b-ii. Capture & save library versions for reproducibility ────────────────
import sys, platform, transformers, peft, PIL, accelerate
try:
    import bitsandbytes as bnb
    bnb_ver = bnb.__version__
except Exception:
    bnb_ver = "n/a"

env_info = {
    "recorded_at":    datetime.datetime.now().isoformat(),
    "python":         sys.version,
    "platform":       platform.platform(),
    "torch":          torch.__version__,
    "cuda":           torch.version.cuda if torch.cuda.is_available() else "n/a",
    "transformers":   transformers.__version__,
    "peft":           peft.__version__,
    "pillow":         PIL.__version__,
    "accelerate":     accelerate.__version__,
    "bitsandbytes":   bnb_ver,
    "numpy":          np.__version__,
    "pandas":         pd.__version__,
}

with open(DRIVE_DIR_PATH + "environment.json", "w") as f:
    json.dump(env_info, f, indent=2)

print("Library versions saved to environment.json")
for k, v in env_info.items():
    print(f"  {k:<16}: {v}")

Library versions saved to environment.json
  recorded_at     : 2026-04-30T21:40:32.917879
  python          : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
  platform        : Linux-6.6.113+-x86_64-with-glibc2.35
  torch           : 2.10.0+cu128
  cuda            : 12.8
  transformers    : 4.57.6
  peft            : 0.18.1
  pillow          : 11.3.0
  accelerate      : 1.13.0
  bitsandbytes    : 0.49.2
  numpy           : 2.0.2
  pandas          : 2.2.2


In [ ]:
# ── 0c. Reproducibility ───────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# Makes CUDA ops deterministic (slight perf cost — fine for a class project)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Global seed set to {SEED}")

Global seed set to 42


In [ ]:
# ── 0d. Device & VRAM ─────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU    : {gpu_name}")
    print(f"VRAM   : {total_vram:.2f} GB")
else:
    print("No GPU found — running on CPU (expect slow training).")

Device : cuda
GPU    : Tesla T4
VRAM   : 14.56 GB


In [ ]:
# ── 0e. BitsAndBytes/QLoRA config ─────────────────────────────────────────────────────────
use_bnb = True # Flag to enable/disable BitsAndBytes quantization

# Determine the dtypes based on bf16 support
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    # If bf16 is supported, use it where applicable
    amp_autocast_dtype = torch.bfloat16
    bnb_compute_dtype = torch.bfloat16
    model_load_dtype = torch.bfloat16
    print("CUDA device supports bfloat16. Using bfloat16 for AMP and QLoRA compute dtype.")
else:
    # If bf16 is NOT supported, use float32 for bnb_4bit_compute_dtype as requested.
    # For AMP autocast and model loading, stick with float16 if cuda is available,
    # otherwise float32 (which is already handled by existing conditionals for CPU).
    amp_autocast_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    bnb_compute_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model_load_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    print("CUDA device does NOT support bfloat16.")
    print(f"  Using {amp_autocast_dtype} for AMP autocast.")
    print(f"  Using {bnb_compute_dtype} for bnb_4bit_compute_dtype (as requested).")
    print(f"  Using {model_load_dtype} for model loading.")

CUDA device supports bfloat16. Using bfloat16 for AMP and QLoRA compute dtype.


## 1. Hyperparameters

In [ ]:
import torch

# ── 1. Define hyperparameters — edit values here, then run this cell ──────────

# Model
MODEL_ID            = "HuggingFaceTB/SmolVLM-500M-Instruct"

# Data
IMG_SIZE            = 256
MAX_SEQ_LEN         = 1024
TRAIN_SUBSET_FRAC   = 0.1    # fraction of train set to use; 1.0 = full dataset

# Training
TRAIN_BATCH         = 4
GRAD_ACCUM          = 4      # effective batch = TRAIN_BATCH * GRAD_ACCUM
EVAL_BATCH          = 16
LR                  = 2e-4
EPOCHS              = 5
WARMUP_RATIO        = 0.05

# LoRA
LORA_RANK           = 8
LORA_ALPHA          = 16
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]


# ── Snapshot to dict (used throughout for logging) ────────────────────────────
cfg = {
    "MODEL_ID":            MODEL_ID,
    "IMG_SIZE":            IMG_SIZE,
    "MAX_SEQ_LEN":         MAX_SEQ_LEN,
    "TRAIN_SUBSET_FRAC":   TRAIN_SUBSET_FRAC,
    "TRAIN_BATCH":         TRAIN_BATCH,
    "GRAD_ACCUM":          GRAD_ACCUM,
    "EVAL_BATCH":          EVAL_BATCH,
    "LR":                  LR,
    "EPOCHS":              EPOCHS,
    "WARMUP_RATIO":        WARMUP_RATIO,
    "LORA_RANK":           LORA_RANK,
    "LORA_ALPHA":          LORA_ALPHA,
    "LORA_DROPOUT":        LORA_DROPOUT,
    "LORA_TARGET_MODULES": LORA_TARGET_MODULES,
    "use_bnb":             use_bnb,
    "amp_autocast_dtype":  str(amp_autocast_dtype).replace("torch.", ""),
    "bnb_compute_dtype":   str(bnb_compute_dtype).replace("torch.", ""),
    "model_load_dtype":    str(model_load_dtype).replace("torch.", ""),
}

# ── Save to disk ──────────────────────────────────────────────────────────────
with open(DRIVE_DIR_PATH + "config.json", "w") as f:
    json.dump(cfg, f, indent=2)

print("config.json saved:")
print(json.dumps(cfg, indent=2))

config.json saved:
{
  "MODEL_ID": "HuggingFaceTB/SmolVLM-500M-Instruct",
  "IMG_SIZE": 256,
  "MAX_SEQ_LEN": 1024,
  "TRAIN_SUBSET_FRAC": 0.1,
  "TRAIN_BATCH": 4,
  "GRAD_ACCUM": 4,
  "EVAL_BATCH": 16,
  "LR": 0.0002,
  "EPOCHS": 5,
  "WARMUP_RATIO": 0.05,
  "LORA_RANK": 8,
  "LORA_ALPHA": 16,
  "LORA_DROPOUT": 0.05,
  "LORA_TARGET_MODULES": [
    "q_proj",
    "v_proj",
    "k_proj",
    "o_proj"
  ],
  "use_bnb": true,
  "amp_autocast_dtype": "bfloat16",
  "bnb_compute_dtype": "bfloat16",
  "model_load_dtype": "bfloat16"
}


## 2. Dataset

In [ ]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

# 'choices' is stored as a JSON string — parse it into a Python list
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train : {len(train_df):,} rows")
print(f"Val   : {len(val_df):,} rows")
print(f"Test  : {len(test_df):,} rows")

# ── Optional training subset ──────────────────────────────────────────────────
if TRAIN_SUBSET_FRAC < 1.0:
    train_df = train_df.sample(frac=TRAIN_SUBSET_FRAC, random_state=SEED).reset_index(drop=True)
    print(f"Using {TRAIN_SUBSET_FRAC:.0%} subset → {len(train_df):,} training rows")

    val_df = val_df.sample(frac=TRAIN_SUBSET_FRAC, random_state=SEED).reset_index(drop=True)
    print(f"Using {TRAIN_SUBSET_FRAC:.0%} subset → {len(val_df):,} validation rows")



Train : 3,109 rows
Val   : 1,048 rows
Test  : 1,008 rows
Using 10% subset → 311 training rows
Using 10% subset → 105 validation rows


In [ ]:
# ── 2b. Prompt template ───────────────────────────────────────────────────────
CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    """
    Build the text prompt for the VLM.
    The <image> token tells the model where to inject vision features.

    Parameters
    ----------
    row            : a single DataFrame row (pd.Series)
    include_answer : if True, append the ground-truth letter (for training)
    """
    context_parts = []
    lecture = row.get("lecture", "")
    hint    = row.get("hint", "")
    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())
    context_str = "\n".join(context_parts)

    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(row["choices"])
    )

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        answer_idx = int(row["answer"])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt


# ── Sanity-check: print one prompt ───────────────────────────────────────────
print("=== SAMPLE TRAINING PROMPT ===")
print(build_prompt(train_df.iloc[0], include_answer=True))
print()
print("=== SAMPLE INFERENCE PROMPT ===")
print(build_prompt(val_df.iloc[0], include_answer=False))

=== SAMPLE TRAINING PROMPT ===
<image>
Context:
People can use the engineering-design process to develop solutions to problems. One step in the process is testing if a potential solution meets the requirements of the design. How can you determine what a test can show? You need to figure out what was tested and what was measured.
Imagine an engineer needs to design a bridge for a windy location. She wants to make sure the bridge will not move too much in high wind. So, she builds a smaller prototype, or model, of a bridge. Then, she exposes the prototype to high winds and measures how much the bridge moves.
First, identify what was tested. A test can examine one design, or it may compare multiple prototypes to each other. In the test described above, the engineer tested a prototype of a bridge in high wind.
Then, identify what the test measured. One of the criteria for the bridge was that it not move too much in high winds. The test measured how much the prototype bridge moved.
Tests ca

In [ ]:
# ── 2c. PyTorch Dataset ───────────────────────────────────────────────────────
class ScienceQADataset(Dataset):
    """
    Loads images from disk and builds prompts on-the-fly.
    is_train=True  → returns full prompt with answer appended (for causal LM loss)
    is_train=False → returns prompt without answer + choice list (for MC scoring)
    """

    def __init__(
        self,
        df: pd.DataFrame,
        data_dir: Path,
        img_size: int = 224,
        is_train: bool = True,
    ):
        self.df        = df.reset_index(drop=True)
        self.data_dir  = data_dir
        self.img_size  = img_size
        self.is_train  = is_train

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        full_path = self.data_dir / rel_path
        img = Image.open(full_path).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        return img

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])

        if self.is_train:
            return {
                "image":  img,
                "text":   build_prompt(row, include_answer=True),
                "answer": int(row["answer"]),
            }
        else:
            return {
                "image":   img,
                "text":    build_prompt(row, include_answer=False),
                "choices": row["choices"],
                "id":      row["id"],
                "answer":  int(row["answer"]) if "answer" in row else -1,
            }


train_ds = ScienceQADataset(train_df, DATA_DIR, IMG_SIZE, is_train=True)
val_ds   = ScienceQADataset(val_df,   DATA_DIR, IMG_SIZE, is_train=False)
test_ds  = ScienceQADataset(test_df,  DATA_DIR, IMG_SIZE, is_train=False)

print(f"train_ds : {len(train_ds):,}")
print(f"val_ds   : {len(val_ds):,}")
print(f"test_ds  : {len(test_ds):,}")

train_ds : 311
val_ds   : 105
test_ds  : 1,008


## 3. Model + QLoRA

In [ ]:
# ── 3a. Processor ─────────────────────────────────────────────────────────────
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.image_processor.max_image_tiles = 1  # disable splitting; 1 tile = base patches only
processor.image_processor.do_image_splitting = False

# Ensure pad token is defined (required for batched inference)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

print(f"Vocab size : {processor.tokenizer.vocab_size:,}")
print(f"Pad token  : '{processor.tokenizer.pad_token}'")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Vocab size : 49,152
Pad token  : '<|im_end|>'


In [ ]:
# ── 3a.i Set Processor Image Size ─────────────────────────────────────────────────────────────
processor.image_processor.max_image_size = {"longest_edge": IMG_SIZE}
processor.image_processor.size = {"longest_edge": IMG_SIZE}

print(f'processor max_image_size: {str(processor.image_processor.max_image_size)}')
print(f'processor size: {str(processor.image_processor.size)}')
print("max_image_tiles  :", processor.image_processor.max_image_tiles)
print(vars(processor.image_processor))

print('Processor ready.')

processor max_image_size: {'longest_edge': 256}
processor size: {'longest_edge': 256}
max_image_tiles  : 1
{'_processor_class': 'Idefics3Processor', 'image_processor_type': 'Idefics3ImageProcessor', 'do_convert_rgb': True, 'do_resize': True, 'size': {'longest_edge': 256}, 'resample': 1, 'do_image_splitting': False, 'max_image_size': {'longest_edge': 256}, 'do_rescale': True, 'rescale_factor': 0.00392156862745098, 'do_normalize': True, 'image_mean': [0.5, 0.5, 0.5], 'image_std': [0.5, 0.5, 0.5], 'do_pad': True, 'max_image_tiles': 1}
Processor ready.


In [ ]:
# ── 3b. 4-bit Quantization Config (QLoRA) ────────────────────────────────────
# NF4 quantization keeps the backbone frozen in 4-bit;
# only the LoRA adapter weights (float16) are trained.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # Normal-Float 4 — best for LLMs
    bnb_4bit_use_double_quant=True,     # Nested quantization saves ~0.4 bit/param
    bnb_4bit_compute_dtype=bnb_compute_dtype,
)

print("BitsAndBytes config ready.")

BitsAndBytes config ready.


In [ ]:
# ── 3c. Load Pretrained HF Model ─────────────────────────────────────────────────────────────
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config if use_bnb else None,
    dtype=model_load_dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)

if not torch.cuda.is_available():
    model = model.to(device)

# Disable caching — incompatible with gradient checkpointing
model.config.use_cache = False

print("Base model loaded.")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Base model loaded.


In [ ]:
# ── 3c.i Set Model Image Sequence Length ─────────────────────────────────────────────────────────────

# Calculate the new image sequence length
new_image_seq_len = 16

# Update the Model Configuration
model.config.image_seq_len = new_image_seq_len

# It's good practice to update it on the processor if the attribute exists
if hasattr(processor, "image_seq_len"):
    processor.image_seq_len = new_image_seq_len
    print(f"processor image_seq_len: {processor.image_seq_len}")

print(f"model image_seq_len: {model.config.image_seq_len}")

print('Base model ready.')

processor image_seq_len: 16
model image_seq_len: 16
Base model ready.


In [ ]:
# ── 3d. LoRA config + PEFT wrapping ──────────────────────────────────────────
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=LORA_TARGET_MODULES,
)

model = get_peft_model(model, lora_cfg)

# ── Parameter count ───────────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable_params:,}  "
      f"({100 * trainable_params / total_params:.3f}%)")
assert trainable_params <= 5_000_000, (
    f"Trainable params {trainable_params:,} exceed the 5M competition limit!"
)

Total params     : 303,911,104
Trainable params : 2,080,768  (0.685%)


## 4. Collate Functions

In [ ]:
# ── 4a. Training collate ──────────────────────────────────────────────────────
# The full prompt (including the answer letter) is fed in.
# Labels = input_ids, but prompt tokens are masked to -100
# so the cross-entropy loss is computed ONLY on the answer token.

def train_collate(batch: list[dict]) -> dict:
    images = [item["image"] for item in batch]
    texts  = [item["text"]  for item in batch]

    # Tokenise the full prompt+answer text together with the image
    encoding = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )

    input_ids      = encoding["input_ids"]           # (B, L)
    attention_mask = encoding["attention_mask"]       # (B, L)

    # ── Build labels: mask everything except the answer token(s) ──────────────
    # Strategy: tokenise just the prompt (no answer), find its length,
    # then mask [0 .. prompt_len-1] → -100 in labels.
    prompt_texts = []
    for item in batch:
        # The prompt without the answer letter (ends with "Answer:")
        full_text   = item["text"]           # "... Answer: B"
        answer_idx  = item["answer"]
        answer_letter = CHOICE_LETTERS[answer_idx]
        # Strip the trailing " X" to recover the bare prompt
        prompt_only = full_text[: full_text.rfind(f" {answer_letter}")]
        prompt_texts.append(prompt_only)

    prompt_enc = processor(
        text=prompt_texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )
    prompt_lens = prompt_enc["attention_mask"].sum(dim=1)  # (B,) actual token counts

    labels = input_ids.clone()
    for i, plen in enumerate(prompt_lens):
        labels[i, :plen] = -100          # mask prompt tokens
    # Also mask padding tokens
    labels[attention_mask == 0] = -100

    encoding["labels"] = labels
    return encoding


# ── 4b. Eval collate ─────────────────────────────────────────────────────────
# For evaluation we keep the raw choices so the MC scoring function can
# build one candidate string per choice and score each independently.

def eval_collate(batch: list[dict]) -> dict:
    """Returns the raw batch dict — MC scorer processes each sample individually."""
    return {
        "images":   [item["image"]   for item in batch],
        "texts":    [item["text"]    for item in batch],
        "choices":  [item["choices"] for item in batch],
        "ids":      [item["id"]      for item in batch],
        "answers":  [item["answer"]  for item in batch],
    }


# ── 4c. DataLoaders ───────────────────────────────────────────────────────────
train_loader = DataLoader(
    train_ds,
    batch_size=TRAIN_BATCH,
    shuffle=True,
    collate_fn=train_collate,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=EVAL_BATCH,
    shuffle=False,
    collate_fn=eval_collate,
    num_workers=2,
    pin_memory=True,
)

test_loader = DataLoader(
    test_ds,
    batch_size=EVAL_BATCH,
    shuffle=False,
    collate_fn=eval_collate,
    num_workers=2,
    pin_memory=True,
)

print(f"Train batches : {len(train_loader):,}")
print(f"Val batches   : {len(val_loader):,}")
print(f"Test batches  : {len(test_loader):,}")

Train batches : 78
Val batches   : 7
Test batches  : 63


## 5. Multiple-Choice Log-Likelihood Scoring

Instead of generating tokens greedily, we score each candidate answer by computing its log-likelihood under the model.  
This is **exact, deterministic, and ~10× faster** than beam-search decoding.

In [ ]:
# ── 5. Batched multiple-choice scoring ───────────────────────────────────────

@torch.inference_mode()
def score_choices_batch(
    model,
    processor,
    images: list[Image.Image],
    prompt_texts: list[str],
    choices_list: list[list[str]],
) -> list[int]:
    """
    For each (image, prompt, choices) triple, score every candidate answer
    by computing the sum of log-probs of the answer token(s) conditioned on
    the prompt.  Returns a list of predicted 0-indexed answer indices.

    Algorithm
    ---------
    For sample i with K_i choices we build K_i full sequences:
        full_i_k = prompt_i + " " + CHOICE_LETTERS[k]
    We tokenise all (sum_i K_i) sequences in one batch call, run a single
    forward pass, and accumulate log-probs of the answer token(s) at their
    positions.  The argmax over K_i gives the predicted answer.

    Parameters
    ----------
    model         : PEFT-wrapped VLM (eval mode)
    processor     : matching AutoProcessor
    images        : list of PIL images  (length = B)
    prompt_texts  : list of prompt strings ending with "Answer:"  (length = B)
    choices_list  : list of choice-string lists  (length = B)

    Returns
    -------
    preds : list[int]  (length = B)
    """
    model.eval()

    all_texts   = []   # full sequences for every (sample, choice)
    all_images  = []   # repeated image per candidate
    sample_idxs = []   # which sample each row belongs to
    choice_idxs = []   # which choice index each row represents

    for s_idx, (prompt, choices, img) in enumerate(zip(prompt_texts, choices_list, images)):
        for c_idx, _ in enumerate(choices):
            answer_letter = CHOICE_LETTERS[c_idx]
            full_seq      = prompt + f" {answer_letter}"
            all_texts.append(full_seq)
            all_images.append(img)
            sample_idxs.append(s_idx)
            choice_idxs.append(c_idx)

    # ── Tokenise all candidates in one batch ──────────────────────────────────
    enc = processor(
        text=all_texts,
        images=all_images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )
    enc = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in enc.items()}

    # ── Also tokenise just the prompts (to find answer-token positions) ───────
    prompt_enc = processor(
        text=[
            prompt_texts[s_idx] for s_idx in sample_idxs
        ],
        images=all_images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )
    prompt_lens = prompt_enc["attention_mask"].sum(dim=1)  # (N_cands,)

    # ── Forward pass ──────────────────────────────────────────────────────────
    with torch.amp.autocast("cuda", dtype=amp_autocast_dtype, enabled=torch.cuda.is_available()):
        logits = model(**enc).logits   # (N_cands, L, V)

    log_probs = F.log_softmax(logits, dim=-1)   # (N_cands, L, V)

    # ── Accumulate log-prob of answer token(s) ────────────────────────────────
    input_ids  = enc["input_ids"]    # (N_cands, L)
    N          = input_ids.shape[0]

    cand_scores = torch.zeros(N, device=logits.device)
    for n in range(N):
        plen = int(prompt_lens[n].item())
        seq_len = int(enc["attention_mask"][n].sum().item())
        # sum log-probs of tokens from plen-1 → seq_len-1
        # (shifted by 1: logits[t] predicts token[t+1])
        for t in range(plen - 1, seq_len - 1):
            next_token = input_ids[n, t + 1]
            cand_scores[n] += log_probs[n, t, next_token]

    # ── Argmax per sample ─────────────────────────────────────────────────────
    B     = len(prompt_texts)
    preds = []
    for s_idx in range(B):
        mask = [i for i, si in enumerate(sample_idxs) if si == s_idx]
        best = int(torch.argmax(cand_scores[mask]).item())
        preds.append(best)

    return preds


print("Multiple-choice scoring function defined.")

Multiple-choice scoring function defined.


## 6. Training Loop

In [ ]:
# ── 6a. Optimizer & Scheduler ─────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=0.01,
)

total_steps   = math.ceil(len(train_loader) / GRAD_ACCUM) * EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Total optimizer steps : {total_steps:,}")
print(f"Warmup steps          : {warmup_steps:,}")

Total optimizer steps : 100
Warmup steps          : 5


In [ ]:
# ── 6b. Helper: validate on the entire val set ────────────────────────────────
def run_validation(model, val_loader) -> tuple[float, float]:
    """
    Returns (val_loss, val_accuracy).
    val_loss     — average NLL of the correct answer token over the val set.
    val_accuracy — fraction of questions answered correctly via MC scoring.
    """
    model.eval()
    total_loss  = 0.0
    total_steps = 0
    correct      = 0
    total       = 0

    for batch in val_loader:
        images   = batch["images"]
        texts    = batch["texts"]
        choices  = batch["choices"]
        answers  = batch["answers"]

        # ── MC accuracy ──────────────────────────────────────────────────────
        preds = score_choices_batch(model, processor, images, texts, choices)
        for pred, gt in zip(preds, answers):
            correct += int(pred == gt)
            total   += 1

        # ── Val loss (NLL on correct answer) ─────────────────────────────────
        # Build full sequences with the ground-truth answer appended
        gt_texts = [
            texts[i] + f" {CHOICE_LETTERS[answers[i]]}"
            for i in range(len(texts))
        ]
        enc = processor(
            text=gt_texts,
            images=images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
        )
        enc = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in enc.items()}

        prompt_enc = processor(
            text=texts,
            images=images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
        )
        prompt_lens = prompt_enc["attention_mask"].sum(dim=1)

        labels = enc["input_ids"].clone()
        for i, plen in enumerate(prompt_lens):
            labels[i, :plen] = -100
        labels[enc["attention_mask"] == 0] = -100
        enc["labels"] = labels

        with torch.inference_mode():
            with torch.amp.autocast("cuda", dtype=amp_autocast_dtype, enabled=torch.cuda.is_available()):
                loss = model(**enc).loss
        total_loss  += loss.item()
        total_steps += 1

    val_loss = total_loss / max(total_steps, 1)
    val_acc  = correct / max(total, 1)
    return val_loss, val_acc

In [ ]:
# ── 6c. Main training loop ────────────────────────────────────────────────────
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available() and amp_autocast_dtype == torch.float16)

train_metrics = []   # will be saved to disk at end
best_val_acc  = -1.0
global_step   = 0
train_start   = time.time()   # ← total training timer

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss     = 0.0
    epoch_steps    = 0
    optimizer.zero_grad()

    for batch_idx, batch_i in enumerate(train_loader):
        # Move tensors to device
        batch = {k: v.to(model.device, non_blocking=True) if torch.is_tensor(v) else v
                 for k, v in batch_i.items()}
        # Explicitly cast attention_mask to bool and ensure contiguity
        # if "attention_mask" in batch and batch["attention_mask"] is not None:
        #     batch["attention_mask"] = batch["attention_mask"].bool().contiguous()

        with torch.amp.autocast("cuda", dtype=amp_autocast_dtype, enabled=torch.cuda.is_available()):
        # with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            outputs = model(**batch)
            loss    = outputs.loss / GRAD_ACCUM

        # Only scale if float16 is used for autocasting
        if amp_autocast_dtype == torch.float16:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(train_loader):
            if amp_autocast_dtype == torch.float16:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()), 1.0
            )
            if amp_autocast_dtype == torch.float16:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

        epoch_loss  += loss.item() * GRAD_ACCUM   # unscale for logging
        epoch_steps += 1

        if (batch_idx + 1) % 100 == 0:
            avg = epoch_loss / epoch_steps
            lr_now = scheduler.get_last_lr()[0]
            print(f"  Epoch {epoch} | step {batch_idx+1}/{len(train_loader)} "
                  f"| loss {avg:.4f} | lr {lr_now:.2e}")

    avg_train_loss = epoch_loss / epoch_steps

    # ── End-of-epoch validation ───────────────────────────────────────────────
    print(f"\nEpoch {epoch} complete — running validation…")
    val_loss, val_acc = run_validation(model, val_loader)

    epoch_record = {
        "epoch":          epoch,
        "train_loss":     round(avg_train_loss, 6),
        "val_loss":       round(val_loss,       6),
        "val_accuracy":   round(val_acc,        6),
        "global_step":    global_step,
        "lr":             scheduler.get_last_lr()[0],
        "epoch_time_sec": round(time.time() - train_start, 1),
    }
    train_metrics.append(epoch_record)

    print(f"\n{'='*60}")
    print(f"Epoch {epoch:2d} | train_loss={avg_train_loss:.4f} "
          f"| val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")
    print(f"{'='*60}\n")

    # ── Save best checkpoint ──────────────────────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained("best_checkpoint")
        processor.save_pretrained("best_checkpoint")
        print(f"  ✓ New best model saved (val_acc={best_val_acc:.4f})")

# ── Save training metrics to disk ─────────────────────────────────────────────
total_training_time = round(time.time() - train_start, 1)
val_wrong_answers   = round((1 - best_val_acc) * len(val_df))
with open(DRIVE_DIR_PATH + "/train_metrics.json", "w") as f:
    json.dump(
        {
            "config":               cfg,
            "best_val_acc":         best_val_acc,
            "total_training_time_sec": total_training_time,
            "val_wrong_answers":    val_wrong_answers,
            "epochs":               train_metrics,
        },
        f,
        indent=2,
    )

print(f"\nTraining complete in {total_training_time:.1f}s — metrics saved to train_metrics.json")


Epoch 1 complete — running validation…

Epoch  1 | train_loss=1.2301 | val_loss=1.0340 | val_acc=0.5048

  ✓ New best model saved (val_acc=0.5048)

Epoch 2 complete — running validation…

Epoch  2 | train_loss=0.7829 | val_loss=1.1384 | val_acc=0.4952


Epoch 3 complete — running validation…

Epoch  3 | train_loss=0.6105 | val_loss=1.2033 | val_acc=0.5048


Epoch 4 complete — running validation…

Epoch  4 | train_loss=0.4799 | val_loss=1.1995 | val_acc=0.5524

  ✓ New best model saved (val_acc=0.5524)

Epoch 5 complete — running validation…

Epoch  5 | train_loss=0.3960 | val_loss=1.2221 | val_acc=0.5524


Training complete in 1327.6s — metrics saved to train_metrics.json


## 7. Inference Example

Loads the best checkpoint and runs it on one validation sample — matches the starter notebook style.

In [ ]:
# ── 7. Inference example on a single val sample ───────────────────────────────
from peft import PeftModel

# Reload base + LoRA adapter from best checkpoint
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config if use_bnb else None,
    dtype=model_load_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)
if not torch.cuda.is_available():
    base_model = base_model.to(device)

inf_model = PeftModel.from_pretrained(base_model, "best_checkpoint")
inf_model.eval()

inf_processor = AutoProcessor.from_pretrained("best_checkpoint")
if inf_processor.tokenizer.pad_token is None:
    inf_processor.tokenizer.pad_token = inf_processor.tokenizer.eos_token

# Pick one sample
sample      = val_df.iloc[0]
sample_img  = Image.open(DATA_DIR / sample["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
sample_prompt = build_prompt(sample, include_answer=False)

preds = score_choices_batch(
    inf_model,
    inf_processor,
    images        = [sample_img],
    prompt_texts  = [sample_prompt],
    choices_list  = [sample["choices"]],
)

pred_idx    = preds[0]
pred_letter = CHOICE_LETTERS[pred_idx]
gt_idx      = int(sample["answer"])
gt_letter   = CHOICE_LETTERS[gt_idx]

print("=== PROMPT ===")
print(sample_prompt)
print()
print(f"Predicted answer : {pred_letter} ({sample['choices'][pred_idx]})")
print(f"Ground-truth     : {gt_letter}  ({sample['choices'][gt_idx]})")
print(f"Correct          : {pred_idx == gt_idx}")

=== PROMPT ===
<image>
Context:
Read the text about spinner dolphins.
Have you ever seen a dolphin spin through the water? How about a dolphin that jumps high above the ocean? If so, you have probably seen a spinner dolphin. These playful dolphins are able to leap into the air and then spin around a few times before crashing back into the water.
Though these dolphins love to play, they spend much of their day swimming peacefully in harbors and resting. This helps them conserve energy for the busy night ahead. When the sun goes down, spinner dolphins hunt for food. At night, the sea animals that the dolphins eat move from the deep ocean toward the surface of the water. After a night of hunting and eating, spinner dolphins are ready to rest in the harbors again.

Question: Based on the text, what is one thing that spinner dolphins do?
Choices:
  A. They spin around in the air.
  B. They hunt for food during the day.
  C. They leap in the air to catch their food.
Answer:

Predicted answer

## 8. Build & Save Submission CSV

In [ ]:
# ── 8. Batched inference over the test set → submission CSV ───────────────────

all_ids    = []
all_preds  = []
infer_start = time.time()   # ← inference timer

print(f"Running inference on {len(test_ds):,} test examples…")

for batch_num, batch in enumerate(test_loader, start=1):
    images   = batch["images"]
    texts    = batch["texts"]
    choices  = batch["choices"]
    ids      = batch["ids"]

    preds = score_choices_batch(inf_model, inf_processor, images, texts, choices)

    all_ids.extend(ids)
    all_preds.extend(preds)

    if batch_num % 20 == 0 or batch_num == len(test_loader):
        done = batch_num * EVAL_BATCH
        print(f"  {min(done, len(test_ds)):,} / {len(test_ds):,} processed")

# ── Build DataFrame ───────────────────────────────────────────────────────────
submission = pd.DataFrame({"id": all_ids, "answer": all_preds})

# Verify ids match the sample submission
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
assert set(submission["id"]) == set(sample_sub["id"]), "ID mismatch!"
submission = submission.set_index("id").loc[sample_sub["id"]].reset_index()  # align order

# ── Save with timestamp ───────────────────────────────────────────────────────
infer_elapsed = round(time.time() - infer_start, 1)
ts       = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = f"{DRIVE_DIR_PATH}/submission_{ts}.csv"
submission.to_csv(out_path, index=False)

# ── Save submission metrics ───────────────────────────────────────────────────
sub_metrics = {
    "submission_file":          out_path,
    "inference_time_sec":       infer_elapsed,
    "total_training_time_sec":  total_training_time,
    "val_wrong_answers":        val_wrong_answers,
    "val_accuracy":             best_val_acc,
    "n_test_examples":          len(submission),
}
with open(f"{DRIVE_DIR_PATH}/submission_metrics_{ts}.json", "w") as f:
    json.dump(sub_metrics, f, indent=2)

print(f"\nSubmission saved → {out_path}")
print(f"Inference time   : {infer_elapsed:.1f}s")
print(f"Val wrong answers: {val_wrong_answers} / {len(val_df)}")
print(f"Shape            : {submission.shape}")
print(f"Answer distribution:")
print(submission["answer"].value_counts().sort_index())
submission.head()

Running inference on 1,008 test examples…
  320 / 1,008 processed
  640 / 1,008 processed
  960 / 1,008 processed
  1,008 / 1,008 processed

Submission saved → /content/drive/MyDrive/Colab Notebooks/run_1//submission_20260430_223826.csv
Inference time   : 634.4s
Val wrong answers: 47 / 105
Shape            : (1008, 2)
Answer distribution:
answer
0    470
1    301
2    161
3     76
Name: count, dtype: int64


,id,answer
0,test_01750,1
1,test_00128,0
2,test_02891,1
3,test_02425,0
4,test_00930,0


## 📓 Experiment Diary — Run End

> **Post-run notes:**

In [ ]:
# Final compute diary stamp — run this at the END of each session
from datetime import datetime
import json, os
stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
with open("diary/compute_diary_p3.txt", "a") as f:
    f.write(json.dumps({"event": "session_end", "when": stamp}) + "\n")
print("⏱️ Session end logged:", stamp)

⏱️ Session end logged: 2026-04-30 22:38:26
